# Description
Calculation of the number of passes using a generated trajectory. Finished on May 18th, 2026. It is the first version it can count by boundary cell. The grid is done by rectangles

In [23]:
%load_ext autoreload
%autoreload 2
%reset -f
%matplotlib widget

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
import matplotlib.pyplot as plt
from shapely import wkt
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
from shapely.geometry import LineString, Point
from shapely.affinity import rotate
import numpy as np
import pandas as pd
import geopandas as gpd
import warnings
from inc import *
from shapely.ops import polygonize, unary_union


# Configuration

In [25]:
tol_deg  = 2.5
street_length = 20
sampling_distance = 5
dx = 1            # Thickness of each vertical bar
angles = np.arange(0, 190, 5)

# Generate the arbitrary pass

In [26]:
# Create a DataFrame with the pass name and geometries
survey_gdf = gpd.GeoDataFrame({
    'Order': [1,2,3,4,5],
    'geometry': [
        LineString([(-800, -800),(0, 0),(-800, 800)]),
        LineString([(800, 800),(0, 700),(-800, 800)]),
        LineString([(800, 800),(0,0 ),(800, -800)]),
        LineString([(775, 775),(0, 675)]),
        LineString([(810, 810),(0, 735)])
    ]
}, geometry='geometry', crs='EPSG:3857')

# Algorithm starts here
## Prepare the surveys

In [27]:
# Compute the union of the survey geometries
survey_union = survey_gdf.unary_union

# Create a GeoDataFrame from the union
survey_union_gdf = gpd.GeoDataFrame(
    {'geometry': [survey_union]},
    crs=survey_gdf.crs
)

# Make an offset of 10 (buffer by 10 units)
survey_union_offset_gdf = gpd.GeoDataFrame(
    {'geometry': [survey_union.buffer(street_length/2, cap_style=2)]},
    crs=survey_gdf.crs,
    geometry='geometry'
)

# Generate the grid

In [28]:
output_gdf = gpd.GeoDataFrame()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for angle in angles:
        #print(f'Processing angle: {angle}')
        # Parameters for evenly spaced thin vertical rectangles (grid "bars")
        bounds = survey_union_offset_gdf.total_bounds
        min_x, min_y, max_x, max_y = map(float, bounds)  # Ensure cast to float
        r_x = np.sqrt(max_x**2 + max_y**2)
        r_y = np.sqrt(min_x**2 + min_y**2)

        # Compute number of columns safely and cast to int
        num_cols = int(np.floor((2*r_x) / sampling_distance)) + 1

        # Compute bar center x-positions
        x_centers = np.linspace(-r_x, r_x, num=num_cols)

        # For each center, form a thin rectangle
        vertical_bars = [
            Polygon([
                (x - dx/2, min_y),
                (x + dx/2, min_y),
                (x + dx/2, max_y),
                (x - dx/2, max_y)
            ])
            for x in x_centers
        ]

        #-- Grid aligned with the base vectors [1.0], 
        grid_lines_gdf = gpd.GeoDataFrame({'geometry': vertical_bars}, crs='EPSG:3857', geometry='geometry')
        grid_lines_gdf['grid_idx'] = grid_lines_gdf.index

        # Define the vector
        vec = np.array([1,0])
        ortho_vec = np.array([0,1])
        # Define a rotation angle in radians (example: 45 degrees)
        theta = np.deg2rad(angle)
        rotation_matrix = np.array([
            [np.cos(theta), -np.sin(theta)],
            [np.sin(theta), np.cos(theta)]
        ])

        # Rotate the vector by the rotation matrix
        rotated_vec = rotation_matrix @ vec
        ortho_vec = rotation_matrix @ ortho_vec

        # Rotate the grid lines accordingly
        vertical_bars_r = [rotate(bar, angle=angle, origin=(0,0)) for bar in vertical_bars]
        grid_lines_gdf_r = gpd.GeoDataFrame({'geometry': vertical_bars_r}, crs='EPSG:3857', geometry='geometry')
        grid_lines_gdf_r['grid_idx'] = grid_lines_gdf_r.index

        #Create the intersection of the grid with the survey offset
        intersection_gdf = gpd.overlay(
            grid_lines_gdf_r, 
            survey_union_offset_gdf, 
            how='intersection', 
            keep_geom_type=False
        )
        intersection_gdf = intersection_gdf.explode(index_parts=True)
        intersection_gdf.set_geometry('geometry', inplace=True)
        intersection_gdf.set_crs('EPSG:3857', inplace=True)

        #Get the bottom points of the intersected rectamgles
        intersection_gdf['bottom_points'] = intersection_gdf['geometry'].apply(get_bottom_points)

        # Get the unit vector along the [1,0] direction or the rotated version
        intersection_gdf['unit_vector'] = intersection_gdf['bottom_points'].apply(safe_unit_vector)

        # Get teh angle between the unit vector and the rotated vector
        intersection_gdf['angle'] = intersection_gdf['unit_vector'].apply(lambda vec: angle_between_vectors(vec, rotated_vec=rotated_vec))

        # Get only those lines which are aligned with the rotated vector
        right_angles = intersection_gdf[np.isclose(intersection_gdf['angle'], 0, atol=tol_deg)]
        if len(right_angles) > 0:
            #print('There is a right angle')
            #Get the cell boundaries
            right_angles['cell_boundary'] = right_angles.apply(lambda row: get_line_from_angle(row, ortho_vec), axis=1)

            #Rename the grid_idx to include the angle
            right_angles['grid_idx'] = right_angles.apply(lambda row: f"{row['grid_idx']}_{angle}", axis=1)
            right_angles.reset_index(drop=True)
            output_gdf = pd.concat([output_gdf, right_angles])

# Remove entries whose cell_boundary length > 1 std above the mean
lengths = output_gdf.cell_boundary.length
mean_length = lengths.mean()
std_length = lengths.std()
filtered_output_gdf = output_gdf[lengths <= (mean_length + std_length)]


#Remove all the intersecting 
#Create the cells
geom = survey_union_offset_gdf.iloc[0].geometry

# All grid lines as one noded multiline (tweak attribute if your geometry column differs)
lines = [getattr(row, "cell_boundary") for row in filtered_output_gdf.itertuples()]
splitters = unary_union(lines)
network = unary_union([geom.boundary, splitters])
cell_polys = [
    poly
    for poly in polygonize(network)
    if geom.contains(poly.representative_point())
]
print(len(lines), "splitters ->", len(cell_polys), "cells")
cell_gdf = gpd.GeoDataFrame({'geometry': cell_polys}, crs=survey_union_offset_gdf.crs)
cell_gdf = cell_gdf.reset_index().rename(columns={'index': 'cell_idx'})
    

1848 splitters -> 1565 cells


## Count the intersections by cell

In [29]:
# For each cell, craete an inner offset to prevent bad counts
cell_offset_gdf = gpd.GeoDataFrame(geometry=cell_gdf.buffer(-0.1), crs=cell_gdf.crs)
cell_boundaries_gdf = gpd.GeoDataFrame(geometry=cell_offset_gdf.boundary, crs=cell_offset_gdf.crs)

#For each cell, partition the polygon into segments along its boundary lines using exterior coordinates. This will help us to get the nummber of boundaries
segment_list = []
for poly in cell_offset_gdf.geometry:
    coords = list(poly.exterior.coords)
    for i in range(len(coords) - 1):
        seg = LineString([coords[i], coords[i+1]])
        segment_list.append(seg)

partitioned_gdf = gpd.GeoDataFrame(geometry=segment_list, crs=cell_offset_gdf.crs)

#Get the intersections of the surveys with the boundaries
points = gpd.overlay(survey_gdf, cell_boundaries_gdf, how='intersection', keep_geom_type=False).explode()
points_buffer = gpd.GeoDataFrame(geometry=points.buffer(0.05), crs=points.crs)

#Get the boundaries that intersect with the survey
points_buffer.reset_index(drop=True, inplace=True)
boundaries = gpd.sjoin(points_buffer, partitioned_gdf, how='right', predicate='intersects')
# Fix: Only cast to int if value is finite (not NA/inf) to avoid IntCastingNaNError
if boundaries['index_left'].notnull().all():
    boundaries['index_left'] = boundaries['index_left'].astype(int)
else:
    # Fill NA with a placeholder (e.g., -1) before casting, or just keep NA if that's acceptable
    boundaries['index_left'] = boundaries['index_left'].fillna(-1).astype(int)
    # Drop rows where index_left == -1
boundaries = boundaries[boundaries['index_left'] != -1]
boundaries.rename(columns={'index_left': 'points_idx'}, inplace=True)
# Get unique geometries from 'boundaries' and put in a GeoDataFrame
unique_geoms = boundaries['geometry'].unique()
unique_gdf = gpd.GeoDataFrame(geometry=list(unique_geoms), crs=boundaries.crs)
unique_gdf.reset_index(drop=True, inplace=True)
joined_gdf_unique = gpd.sjoin(unique_gdf, cell_gdf, how='right', predicate='intersects')
boundary_counts = joined_gdf_unique.groupby('cell_idx').size().reset_index(name='boundaries')

# Get the number of intersections per cell
joined_gdf = gpd.sjoin(points, cell_gdf, how='right', predicate='intersects')
# Get intersection counts by cell
intersection_counts = joined_gdf.groupby('cell_idx').size().reset_index(name='intersections')
intersection_boundaties = intersection_counts.merge(boundary_counts, left_on='cell_idx', right_on='cell_idx', how='left')

#Get the output
cell_summary = cell_gdf.merge(intersection_boundaties, left_on='cell_idx', right_on='cell_idx', how='left')
cell_summary['passes'] = cell_summary['intersections'] / cell_summary['boundaries']

/tmp/ipykernel_6495/3325188014.py:16: FutureWarning: Currently, index_parts defaults to True, but in the future, it will default to False to be consistent with Pandas. Use `index_parts=True` to keep the current behavior and True/False to silence the warning.
  points = gpd.overlay(survey_gdf, cell_boundaries_gdf, how='intersection', keep_geom_type=False).explode()


In [32]:
# Use geopandas's dissolve to aggregate geometries by the 'passes' column
# This will produce a GeoDataFrame with one row per unique 'passes' value, with multipolygons where appropriate.

# Create a new GeoDataFrame with necessary columns
cell_info_nonan = cell_summary.dropna(subset=['passes']).copy()

dissolved = cell_info_nonan.dissolve(by='passes')

# Ensure the dissolved result is a GeoDataFrame indexed by 'passes'
aggregated_cells_by_pass_gdf = dissolved.reset_index()[['passes', 'geometry']]

aggregated_cells_by_pass_gdf  # This GeoDataFrame has each unique pass value and its aggregated (Multi)Polygon geometry

,passes,geometry
0,0.400000,"POLYGON ((-807.071 792.929, -800.000 800.000, ..."
1,0.666667,"POLYGON ((119.877 725.062, 124.845 725.683, 12..."
2,1.000000,"MULTIPOLYGON (((-780.070 -765.928, -776.532 -7..."
3,1.333333,"MULTIPOLYGON (((747.737 781.565, 752.702 782.2..."
4,2.000000,"MULTIPOLYGON (((490.822 790.489, 495.804 790.9..."
5,2.333333,"POLYGON ((764.826 815.860, 774.789 816.782, 77..."
6,3.000000,"POLYGON ((784.752 817.705, 789.734 818.166, 79..."
